# Advertising Benchmark

Goal: measure whether MF-REINFORCE learns advertising policies that increase the informed/customer population while controlling advertising cost.

**Environment Basics**

- State space: $\mathcal{X}=\{N,C\}$, encoded as `{0, 1}` for non-customer and customer.
- Action space: $\mathcal{A}=\{NO\_AD,AD\}$, encoded as `{0, 1}`.
- Population law: $\mu_t=(1-p_t,p_t)$, where $p_t=\mu_t(C)$ is the customer fraction.
- Policy input: normalized time and current customer fraction $p_t$; the policy outputs the advertisement probability.
- Main task: increase $p_t$ quickly, but avoid paying advertising cost when organic/current adoption is already high.

The finite-state population follows

$$
\mu_{t+1}(y)=\sum_x \mu_t(x)\sum_a \pi_\theta(a\mid x,\mu_t)P(y\mid x,a,\mu_t),
$$

and the reported value is the discounted population reward

$$
J(\theta)=\sum_{t=0}^{T}\gamma^t\sum_x \mu_t(x)r(x,a_t,\mu_t),
\qquad
r(x,a,\mu)=x-c_{ad}a.
$$

The main application statistic is the customer fraction $p_t=\mu_t(1)$ together with cumulative population gain and advertising expenditure.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mfc.experiments import notebook_helpers as nh

ENV_NAME = "advertising"
BASE_DIR = ROOT / "runs" / "notebook_bundles" / ENV_NAME
PRESET = "smoke"
QUICK = PRESET == "smoke"
RUN_MISSING = True
FORCE_REBUILD = False
EXTENDED = True

In [ ]:
bundle = (
    nh.ensure_discrete_benchmark_bundle(ENV_NAME, BASE_DIR, quick=QUICK, force=FORCE_REBUILD, extended=EXTENDED, preset=PRESET)
    if RUN_MISSING
    else nh.bundle_paths(ENV_NAME, BASE_DIR)
)
bundle

## Figure Coverage

Goal: show which requested results from `docs/figures.md` are currently produced by this notebook and which artifacts support them.

The tables are an audit layer, not an estimator. A row maps a desired result family to the command or study that generates its data and to the notebook helper that renders it.

In [ ]:
nh.figure_checklist(ENV_NAME)

In [ ]:
nh.figure_coverage_matrix(ENV_NAME)

## Training: Simplex vs Logits

Goal: compare the optimization traces of the two finite-state perturbation geometries.

The value/objective plot estimates $J(\theta_k)$ at training episode $k$. The gradient plot tracks $\|\widehat g_k\|_2$, where $\widehat g_k$ is the MF-REINFORCE gradient estimate used by Adam.

Simplex uses affine law perturbations on the simplex; logits uses logistic-normal perturbations in logit coordinates. The curves should be compared using the same saved train/evaluation budget.

In [ ]:
histories = nh.load_training_histories(bundle)
nh.plot_training_comparison(histories)

## Application Diagnostics

Goal: show whether the policy creates adoption/customer growth efficiently.

Reference: the table and dashed curves use the finite-horizon dynamic-programming oracle $q_t^*(p)$ on a customer-fraction grid; the policy heatmap also reports the infinite-horizon threshold reference when available.

The key curve is

$$
p_t=\mu_t(1),
$$

with horizontal target levels at 50% and 80%. The objective decomposition compares cumulative population gain with cumulative advertising cost, and the finite-population panel reports the approximation gap as particle count changes.

In [ ]:
application = nh.load_application_data(bundle)
display(nh.reference_solution_table(ENV_NAME, application))
nh.plot_population_flow(application, ENV_NAME)
nh.plot_time_metrics(application, ENV_NAME)
nh.plot_policy_heatmaps(application, ENV_NAME)
nh.plot_discrete_application_details(application, ENV_NAME)

## Universal Diagnostics

Goal: validate the estimator chain before interpreting optimization performance.

The perturbation plots measure empirical geometry,

$$
d(M^\lambda,\mu),
$$

including quantile bands and local log-log slopes. The functional-law plots study

$$
\Gamma(M^\lambda)=(F_1(M^\lambda),\ldots,F_k(M^\lambda)),
\qquad
\frac{\Gamma(M^\lambda)-\Gamma(\mu)}{\lambda},
$$

which is the induced law of population signatures. The score plots check

$$
S_{t,\lambda}^\theta=\nabla_\theta\log q_{t,\lambda}^\theta(M_t),
\qquad \mathbb{E}[S_{t,\lambda}^\theta]\approx 0,
$$

and the gradient plots report bias, variance, MSE, norm ratio, and cosine agreement for an estimator $\widehat g$ against an oracle or reference gradient $g$:

$$
\operatorname{MSE}=\mathbb{E}\|\widehat g-g\|_2^2,
\qquad
\cos(\widehat g,g)=\frac{\widehat g\cdot g}{\|\widehat g\|_2\|g\|_2}.
$$

The sensitivity plots track errors in $D_t=\partial_\theta\Gamma(\mu_t^\theta)$, or in the finite-state case $D_t=\partial_\theta\mu_t^\theta$.

In [ ]:
diagnostics = nh.load_diagnostic_data(bundle)
nh.plot_perturbation_geometry(diagnostics)
nh.plot_perturbation_slopes(diagnostics)
nh.plot_functional_law(diagnostics)
nh.plot_functional_signature_means(diagnostics)
nh.plot_score_validation(diagnostics)
nh.plot_score_coordinate_diagnostics(diagnostics)
nh.plot_gradient_validation(diagnostics)
nh.plot_gradient_error_decomposition(diagnostics)
nh.plot_sensitivity_validation(diagnostics)
nh.plot_sensitivity_heatmap(diagnostics)

## Scaling, Budget, And Optimization Summaries

Goal: measure how estimator quality and optimization performance change with simulator budget, auxiliary budget, horizon, and training time.

The budget heatmap varies main and auxiliary samples $(B,n)$ under the approximate cost model

$$
C\approx C_{main}B+C_{aux}n.
$$

The horizon plot studies how gradient error changes with $T$, and the optimization plots compare objective or cost gaps against iteration count, runtime, and simulator-call budget.

In [ ]:
studies = nh.load_study_data(bundle)
grid_metrics = nh.load_study_grid_metrics(bundle)
optimization_history = nh.load_optimization_history(bundle)
nh.plot_budget_and_horizon(studies)
nh.plot_budget_pareto(studies, grid_metrics)
nh.plot_optimization_history(optimization_history)
nh.plot_optimization_summary(studies)

## Raw Tables For Custom Figures

Goal: expose the underlying CSV/JSON artifacts used by the plots so paper figures can be restyled or recomputed without rerunning training.

These tables are not new estimators. They are the saved values for population flows, policies, diagnostics, study grids, histories, and final metrics.

In [ ]:
application["simplex"]["time_metrics"].head(), diagnostics["simplex"]["gradient"].head(), studies["budget"].head()